# Reinforcement Pre-Training (RPT) | OmniMATH P70 | Qwen2.5-1.5B | L4

Pre-trains Qwen2.5-1.5B on OmniMATH using RL with dense reward (Appendix A, Dong et al. 2025).
Entropy filtering at P70 selects moderately difficult token positions.
Produces the RPT checkpoint used in rpt_grpo.ipynb and rpt_intuitor.ipynb.

Result: NTP Accuracy 35.5% | Perplexity 2.82

> Selecione **L4** antes de rodar.

## Celula 0 -- GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram:.1f} GB')
    assert vram >= 20
    print('OK GPU verificada')

Sat Jun  6 19:11:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Celula 1 -- Instalacao

In [ ]:
!pip install -q transformers==4.51.0 trl==0.17.0 datasets==3.2.0 accelerate==1.4.0 peft==0.15.1
!pip install -q bitsandbytes sentencepiece tqdm
print('OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.0/411.0 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
OK


## Celula 2 -- Config global

In [ ]:
import os, re, json, warnings, logging, math
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig, TaskType
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.WARNING)

# ── PRE-PROCESSAMENTO ─────────────────────────────────────────────────────
N_PROBLEMS     = 1000
MAX_TOKENS     = 512
CONTEXT_WINDOW = 256
PERCENTILE     = 70        # P70 -- top 30% tokens mais dificeis
TOP_K          = 16        # paper usa top-16
DATASET_FILE   = 'rpt_dataset_p70.jsonl'

# ── TREINO ────────────────────────────────────────────────────────────────
MODEL_NAME      = 'Qwen/Qwen2.5-1.5B'
N_TRAIN         = 750
SEED            = 42
LEARNING_RATE   = 1e-6
LORA_R          = 16
LORA_ALPHA      = 32
NUM_EPOCHS      = 1
NUM_GENERATIONS = 8
TEMPERATURE     = 0.8
BETA            = 0.0
OUTPUT_DIR      = 'qwen-1.5b-rpt-v3-omnimath'

# Prompt template v0 do paper (Apendice D) -- sem system prompt
RPT_PROMPT = (
    "Complete the given text under '### Context' by predicting the next token, "
    "and wrap it in \\boxed{{}}. "
    "Please reason step by step to find the most probable next token as the final answer, "
    "and enclose it in \\boxed{{}} "
    "(note: the token may begin with a space, e.g., \\boxed{{ para}} or \\boxed{{ =}}; "
    "do not use \\text{{}}).\n\n"
    "### Context\n{context}"
)

set_seed(SEED)
print(f'Modelo      : {MODEL_NAME}')
print(f'Percentil   : {PERCENTILE} (top {100-PERCENTILE}% tokens mais dificeis)')
print(f'Reward      : DENSE (Apendice A do paper)')
print(f'Beta (KL)   : {BETA}')
print(f'G rollouts  : {NUM_GENERATIONS}')
print(f'Temperatura : {TEMPERATURE}')

Modelo      : Qwen/Qwen2.5-1.5B
Percentil   : 70 (top 30% tokens mais dificeis)
Reward      : DENSE (Apendice A do paper)
Beta (KL)   : 0.0
G rollouts  : 8
Temperatura : 0.8


## Celula 3 -- Google Drive

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH    = '/content/drive/MyDrive/intuitor-ic'
    DATASET_PATH = f'{BASE_PATH}/{DATASET_FILE}'
    SAVE_DIR     = f'{BASE_PATH}/{OUTPUT_DIR}'
else:
    DATASET_PATH = f'/content/{DATASET_FILE}'
    SAVE_DIR     = f'/content/{OUTPUT_DIR}'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Dataset path: {DATASET_PATH}')
print(f'Checkpoint  : {SAVE_DIR}')
print(f'Dataset ja existe: {os.path.exists(DATASET_PATH)}')
print()
print('Se dataset ja existe, pode pular as Celulas 4-7 e ir direto para a Celula 8.')

Mounted at /content/drive
Dataset path: /content/drive/MyDrive/intuitor-ic/rpt_dataset_p70.jsonl
Checkpoint  : /content/drive/MyDrive/intuitor-ic/qwen-1.5b-rpt-v3-omnimath
Dataset ja existe: False

Se dataset ja existe, pode pular as Celulas 4-7 e ir direto para a Celula 8.


## Celula 4 -- Carrega OmniMATH e modelo proxy

*Pule esta celula se o dataset ja existir no Drive.*

In [ ]:
if os.path.exists(DATASET_PATH):
    print(f'Dataset ja existe: {DATASET_PATH}')
    print('Pulando carregamento do modelo proxy.')
    print('Va para a Celula 8.')
else:
    print('Carregando OmniMATH...')
    raw_omni = load_dataset('KbsdJames/Omni-MATH', split='test', trust_remote_code=True)
    raw_omni = raw_omni.shuffle(seed=SEED).select(range(min(N_PROBLEMS, len(raw_omni))))
    print(f'OK {len(raw_omni)} problemas')

    print(f'\nCarregando modelo proxy: {MODEL_NAME}')
    proxy_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True,
    )
    proxy_model.eval()
    proxy_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if proxy_tok.pad_token is None:
        proxy_tok.pad_token = proxy_tok.eos_token
    vram = torch.cuda.memory_allocated() / 1e9
    print(f'OK | {vram:.2f} GB VRAM')

Carregando OmniMATH...


README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/4428 [00:00<?, ? examples/s]

OK 1000 problemas

Carregando modelo proxy: Qwen/Qwen2.5-1.5B


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

OK | 3.09 GB VRAM


## Celula 5 -- Calcula entropias (1 forward pass por problema)

*Pule esta celula se o dataset ja existir no Drive.*

In [ ]:
if os.path.exists(DATASET_PATH):
    print('Dataset ja existe -- pulando calculo de entropias.')
else:
    def compute_entropies(text, tok, mdl, max_tok=MAX_TOKENS, ctx_win=CONTEXT_WINDOW, topk=TOP_K):
        ids     = tok.encode(text, add_special_tokens=False)
        if len(ids) < 4: return []
        seq_len = min(len(ids), max_tok + 1)
        inp_ids = ids[:seq_len]
        tensor  = torch.tensor([inp_ids], dtype=torch.long).to(mdl.device)
        with torch.no_grad():
            logits = mdl(tensor).logits[0]  # (seq, vocab)
        results = []
        for pos in range(1, seq_len):
            pk = F.softmax(logits[pos-1], dim=-1).topk(topk).values
            pk = pk / pk.sum()
            h  = float(-(pk * torch.log(pk + 1e-10)).sum().item())
            ctx_start = max(0, pos - ctx_win)
            results.append({
                'position'         : pos,
                'entropy'          : h,
                'context_ids'      : inp_ids[ctx_start:pos],
                'target_token_id'  : inp_ids[pos],
                'target_token_str' : tok.decode([inp_ids[pos]]),
            })
        return results

    print(f'Calculando entropias em {N_PROBLEMS} problemas...')
    all_records = []
    for idx in tqdm(range(len(raw_omni)), desc='Problemas'):
        ex   = raw_omni[idx]
        text = ex['problem'] + (' ' + ex.get('solution','') if ex.get('solution') else '')
        for p in compute_entropies(text, proxy_tok, proxy_model):
            all_records.append({**p, 'problem_idx': idx})

    entropies_arr = np.array([r['entropy'] for r in all_records])
    print(f'OK {len(all_records):,} posicoes | P70={np.percentile(entropies_arr, PERCENTILE):.3f}')

Calculando entropias em 1000 problemas...


Problemas: 100%|██████████| 1000/1000 [02:28<00:00,  6.73it/s]


OK 368,152 posicoes | P70=1.125


## Celula 6 -- Filtra por P70 e salva dataset

*Pule esta celula se o dataset ja existir no Drive.*

In [ ]:
if os.path.exists(DATASET_PATH):
    print('Dataset ja existe -- pulando filtragem.')
else:
    threshold = float(np.percentile(entropies_arr, PERCENTILE))
    print(f'Threshold P{PERCENTILE}: {threshold:.4f}')

    n_saved = 0
    with open(DATASET_PATH, 'w', encoding='utf-8') as f:
        for rec in all_records:
            if rec['entropy'] < threshold:
                continue
            ctx_str = proxy_tok.decode(
                rec['context_ids'], skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )
            entry = {
                'context'         : ctx_str,
                'target_token'    : rec['target_token_str'],
                'target_token_id' : rec['target_token_id'],
                'entropy'         : round(rec['entropy'], 4),
                'problem_idx'     : rec['problem_idx'],
                'position'        : rec['position'],
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            n_saved += 1

    print(f'OK {n_saved:,} exemplos salvos em {DATASET_PATH}')

    # Libera memoria do proxy antes do treino
    del proxy_model, proxy_tok
    torch.cuda.empty_cache()
    print('OK Memoria do proxy liberada')

Threshold P70: 1.1250
OK 110,544 exemplos salvos em /content/drive/MyDrive/intuitor-ic/rpt_dataset_p70.jsonl
OK Memoria do proxy liberada


## Celula 7 -- Verifica dataset

In [ ]:
records_all = [json.loads(l) for l in open(DATASET_PATH)]
print(f'OK Dataset carregado: {len(records_all):,} exemplos')

ents = np.array([r['entropy'] for r in records_all])
print(f'Entropia -- min: {ents.min():.3f} | media: {ents.mean():.3f} | max: {ents.max():.3f}')
print(f'\nExemplo:')
print(f'  target  : {records_all[0]["target_token"]!r}')
print(f'  context : ...{records_all[0]["context"][-60:]!r}')

OK Dataset carregado: 110,544 exemplos
Entropia -- min: 1.125 | media: 1.730 | max: 2.750

Exemplo:
  target  : ' integers'
  context : ...'Choose positive'


## Celula 8 -- Dataset de treino

Formata com o prompt template v0 do paper (sem system prompt).
Preserva o campo `context` para uso na reward function.

In [ ]:
def make_conversation(example):
    return {
        'prompt'         : [{'role': 'user', 'content': RPT_PROMPT.format(context=example['context'])}],
        'target_token'   : example['target_token'],
        'target_token_id': example['target_token_id'],
        'context'        : example['context'],
    }


import random
random.seed(SEED)
records = list(records_all)
random.shuffle(records)
train_records = records[:N_TRAIN]

KEEP = ['prompt', 'target_token', 'target_token_id', 'context']
train_dataset = Dataset.from_list(train_records)
train_dataset = train_dataset.map(make_conversation, batched=False)
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c not in KEEP]
)

print(f'OK Dataset de treino: {len(train_dataset)} exemplos')
print(f'Colunas: {train_dataset.column_names}')

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

OK Dataset de treino: 750 exemplos
Colunas: ['context', 'target_token', 'target_token_id', 'prompt']


## Celula 9 -- Modelo de treino e Tokenizer

In [ ]:
print(f'Carregando modelo de treino: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True,
)
model.config.pad_token_id = model.config.eos_token_id

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram = torch.cuda.memory_allocated() / 1e9
print(f'OK | {vram:.2f} GB VRAM')

Carregando modelo de treino: Qwen/Qwen2.5-1.5B
OK | 3.10 GB VRAM


## Celula 10 -- Dense Reward (Apendice A do paper)

```
reward = 1.0                             se acertou
reward = P(predicted_token | context; th) se errou
```

Provides continuous gradient signal at every training step.


In [ ]:
def extract_pred(text):
    zones = []
    k = text.rfind('</think>')
    if k != -1: zones.append(text[k:])
    zones.append(text)
    for z in zones:
        idx = z.rfind(r'\boxed{')
        if idx == -1: idx = z.rfind('boxed{')
        if idx == -1: continue
        s = z.find('{', idx) + 1
        d = 1
        for i, c in enumerate(z[s:], s):
            if c == '{': d += 1
            elif c == '}':
                d -= 1
                if d == 0: return z[s:i]
    return ''


def reward_rpt_dense(completions, prompts, target_token,
                     target_token_id, context, **kwargs):
    predictions  = [extract_pred(c[0]['content']) for c in completions]
    correct_mask = [p.strip() == t.strip() for p, t in zip(predictions, target_token)]

    if all(correct_mask):
        return [1.0] * len(completions)

    enc = tokenizer(
        list(context), return_tensors='pt', padding=True,
        truncation=True, max_length=512
    ).to(model.device)
    with torch.no_grad():
        probs = F.softmax(model(**enc).logits[:, -1, :], dim=-1)

    rewards = []
    for i, (ok, pred, tid) in enumerate(zip(correct_mask, predictions, target_token_id)):
        if ok:
            rewards.append(1.0)
        else:
            pids = tokenizer.encode(pred, add_special_tokens=False) if pred else []
            pid  = pids[0] if pids else tid
            rewards.append(float(probs[i, pid].item()))
    return rewards


# Teste
t_compl = [
    [{'role': 'assistant', 'content': r'\boxed{ size}'}],
    [{'role': 'assistant', 'content': r'\boxed{and}'}],
    [{'role': 'assistant', 'content': r'\boxed{the}'}],
]
t_ctx  = [train_dataset[0]['context']] * 3
t_tgt  = [' size'] * 3
t_ids  = [train_dataset[0]['target_token_id']] * 3
t_r    = reward_rpt_dense(t_compl, None, t_tgt, t_ids, context=t_ctx)

print(f'Teste dense reward:')
print(f'  Acerto (" size") : {t_r[0]:.4f} <- deve ser 1.0')
print(f'  Erro   ("and")   : {t_r[1]:.4f} <- P("and"|ctx)')
print(f'  Erro   ("the")   : {t_r[2]:.4f} <- P("the"|ctx)')
assert t_r[0] == 1.0
assert t_r[1] > 0.0
assert t_r[1] != t_r[2]
std_t = float(torch.tensor(t_r).std().item())
print(f'  reward_std       : {std_t:.4f} <- deve ser > 0')
print('OK Dense reward verificado')

Teste dense reward:
  Acerto (" size") : 1.0000 <- deve ser 1.0
  Erro   ("and")   : 0.0000 <- P("and"|ctx)
  Erro   ("the")   : 0.0000 <- P("the"|ctx)
  reward_std       : 0.5773 <- deve ser > 0
OK Dense reward verificado


## Celula 11 -- LoRA + GRPOConfig

In [ ]:
peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','up_proj','down_proj','gate_proj'],
    task_type=TaskType.CAUSAL_LM, lora_dropout=0.05, bias='none',
)

training_args = GRPOConfig(
    output_dir=SAVE_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True, fp16=False,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=512,
    max_completion_length=512,
    temperature=TEMPERATURE,
    beta=BETA,
    epsilon=0.2,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    use_vllm=False,
    report_to='none',
    seed=SEED,
)

steps = N_TRAIN // training_args.per_device_train_batch_size
print(f'OK | ~{steps} steps | ~8-12h na L4')

OK | ~750 steps | ~8-12h na L4


## Celula 12 -- Treino RPT


In [ ]:
class EnableGradsCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        model.enable_input_require_grads()
        print('OK enable_input_require_grads')


print('='*60)
print(f'TREINO RPT v3 | {MODEL_NAME}')
print(f'Dataset: OmniMATH P70 ({N_TRAIN} exemplos)')
print(f'Reward : dense (1.0 certo | P(pred|ctx) errado)')
print(f'G={NUM_GENERATIONS} | T={TEMPERATURE} | beta={BETA} | lr={LEARNING_RATE}')
print('='*60)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_rpt_dense,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[EnableGradsCallback()],
)

trainer.train()

print('\nOK Treino concluido!')
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Modelo salvo em: {SAVE_DIR}')

TREINO RPT v3 | Qwen/Qwen2.5-1.5B
Dataset: OmniMATH P70 (750 exemplos)
Reward : dense (1.0 certo | P(pred|ctx) errado)
G=8 | T=0.8 | beta=0.0 | lr=1e-06


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


OK enable_input_require_grads


Step,Training Loss
5,0.000000
10,-0.000000
15,-0.000000
20,-0.000000
25,-0.000000
30,-0.000000
35,-0.000000
40,0.000000
45,0.000000
50,-0.000000



OK Treino concluido!
Modelo salvo em: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-rpt-v3-omnimath


## Celula 13 -- Avaliacao NTP + Perplexidade

In [ ]:
    N_NTP_EVAL  = 200
N_PERP_EVAL = 100
model.eval()

# NTP ACCURACY
print(f'{"="*60}')
print(f'NTP ACCURACY ({N_NTP_EVAL} exemplos)')
print(f'{"="*60}')
ntp_eval = records_all[-N_NTP_EVAL:]
correct  = 0
for i, rec in enumerate(ntp_eval):
    inp = tokenizer(rec['context'], return_tensors='pt',
                    truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        pred = int(model(**inp).logits[0, -1, :].argmax().item())
    if pred == rec['target_token_id']: correct += 1
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/{N_NTP_EVAL}] acc: {correct/(i+1):.4f} ({correct/(i+1)*100:.2f}%)')
rpt3_ntp = correct / N_NTP_EVAL
print(f'NTP Accuracy: {rpt3_ntp*100:.2f}%')

# PERPLEXIDADE
print(f'\n{"="*60}')
print(f'PERPLEXIDADE ({N_PERP_EVAL} exemplos)')
print(f'{"="*60}')
raw_eval = load_dataset('KbsdJames/Omni-MATH', split='test', trust_remote_code=True)
raw_eval = raw_eval.shuffle(seed=SEED).select(range(N_TRAIN, N_TRAIN+N_PERP_EVAL))
def tok_p(ex):
    t = tokenizer(f"Problem: {ex.get('problem','')}\nSolution: {ex.get('solution','')}",
                  truncation=True, max_length=512, padding='max_length', return_tensors=None)
    l = t['input_ids'].copy()
    for i,m in enumerate(t['attention_mask']):
        if m==0: l[i]=-100
    t['labels']=l; return t
pd = raw_eval.map(tok_p, batched=False, remove_columns=raw_eval.column_names)
ldr= DataLoader(pd, batch_size=4, shuffle=False,
    collate_fn=lambda x: {k: torch.tensor([d[k] for d in x]).to(model.device)
                          for k in ['input_ids','attention_mask','labels']})
tl, tt = 0.0, 0
with torch.no_grad():
    for b in ldr:
        o=model(**b); n=(b['labels']!=-100).sum().item()
        tl+=o.loss.item()*n; tt+=n
rpt3_ppl = math.exp(tl/tt)
print(f'Perplexidade: {rpt3_ppl:.2f}')

# COMPARACAO FINAL
print(f'\n{"="*60}')
print('COMPARACAO FINAL')
print(f'{"="*60}')
bl = {'BASE':(23.5,2.65),'SPT':(26.5,2.47),'RPT v1':(23.5,2.77),
      'RPT v2 ckpt250':(25.5,2.82),'RLP':(24.5,2.82)}
print(f'{"Metodo":<18} {"NTP":>8} {"Ppl":>7}')
print('-'*36)
for n,(nt,pp) in bl.items():
    print(f'{n:<18} {nt:>7.2f}% {pp:>7.2f}')
print(f'{"RPT v3":<18} {rpt3_ntp*100:>7.2f}% {rpt3_ppl:>7.2f}')
print('='*60)

NTP ACCURACY (200 exemplos)
  [50/200] acc: 0.4400 (44.00%)
  [100/200] acc: 0.3800 (38.00%)
  [150/200] acc: 0.3667 (36.67%)
  [200/200] acc: 0.3550 (35.50%)
NTP Accuracy: 35.50%

PERPLEXIDADE (100 exemplos)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Perplexidade: 2.82

COMPARACAO FINAL
Metodo                  NTP     Ppl
------------------------------------
BASE                 23.50%    2.65
SPT                  26.50%    2.47
RPT v1               23.50%    2.77
RPT v2 ckpt250       25.50%    2.82
RLP                  24.50%    2.82
RPT v3               35.50%    2.82
